# 03 — ResNet-1D Training

Training the residual 1D network on MIT-BIH beat segments.  
ResNet-1D adds skip connections to improve gradient flow through deep layers,
a key innovation from the reference paper (arXiv:2303.03660).

**Architecture:** Stem + 4 residual stages (64→128→256→512 channels)

In [ ]:
import os, sys
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))
from src.dataset import get_dataloaders
from src.models.resnet_1d import ResNet1D
from src.train import train_epoch, eval_epoch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
X = np.load('../data/X.npy')
y = np.load('../data/y.npy')
train_loader, val_loader, test_loader, class_weights = get_dataloaders(X, y, batch_size=64)

In [ ]:
EPOCHS = 50
model = ResNet1D(n_leads=2, n_classes=5).to(device)
print(f'ResNet-1D parameters: {sum(p.numel() for p in model.parameters()):,}')

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val_acc = 0

for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    vl_loss, vl_acc = eval_epoch(model, val_loader, criterion, device)
    scheduler.step()

    for k, v in zip(['train_loss','val_loss','train_acc','val_acc'],
                    [tr_loss, vl_loss, tr_acc, vl_acc]):
        history[k].append(v)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save({'model_state': model.state_dict(), 'val_acc': vl_acc},
                   '../results/baseline/resnet_best.pth')

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | Loss {tr_loss:.4f}/{vl_loss:.4f} | '
              f'Acc {tr_acc*100:.2f}%/{vl_acc*100:.2f}%')

print(f'Best val acc: {best_val_acc*100:.2f}%')

In [ ]:
epochs_range = range(1, EPOCHS+1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs_range, history['train_loss'], label='Train', color='#4CAF50')
axes[0].plot(epochs_range, history['val_loss'],   label='Val',   color='#F44336')
axes[0].set_title('Loss — ResNet-1D', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, [a*100 for a in history['train_acc']], label='Train', color='#4CAF50')
axes[1].plot(epochs_range, [a*100 for a in history['val_acc']],   label='Val',   color='#F44336')
axes[1].set_title('Accuracy — ResNet-1D', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/baseline/resnet_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

ckpt = torch.load('../results/baseline/resnet_best.pth', map_location=device)
model.load_state_dict(ckpt['model_state'])
_, test_acc = eval_epoch(model, test_loader, criterion, device)
print(f'Test Accuracy: {test_acc*100:.2f}%')